# mesh04 — 몸체 만들기: 스펙 숫자가 드론 모양이 되기까지

> ⚠ **이 노트북은 생성물이다.** 수정은 `report_mesh/src/make_mesh04.py` 에서 하고
> 재실행할 것(`.ipynb` 를 직접 고치면 다음 빌드에서 사라진다).

**이 편이 답하는 질문** — 공표 제원의 숫자가 어떤 순서로 드론 형상이 되는가 — 그리고 그 과정에서 형상이 «어디서 정확하고 어디서 대가를 치르는가».

**무엇을 근거로 하는가** (본문 수치는 손으로 적지 않고 아래 **원장** — 검사기가 낸 측정 기록 파일 — 에서 주입한다)

| 원장 | 무엇이 들어 있나 |
|---|---|
| `outputs/mesh_inspect_body_arms_0816.json` | 동체·팔·다리·모터 벨 전수 실측(10종) + matrice4e 공식 CAD 정정 착지 검증 |
| `outputs/mesh_inspect_gimbal_sensors_0816.json` | 짐벌·카메라·센서 검사(10종) — 부착 게이트 A~D |
| `outputs/meshfix_matrice4e.json` | DJI 공식 STEP 대조 정정 명세 14건(matrice4e) |
| `docs/MESH_AUDIT_0816.md` | 적대적 감사 — 발견·반증·수리 우선순위(§⑤) |
| `report_mesh/outputs/mesh_verify_canon_0817.json` | ⭐**정본 판** 기하·치수·대칭 원장(A/B/C/D/F/G) — 이 편 수치의 기본 원장 |
| `report_mesh/outputs/mesh_canon_0817.json` | 정본 판 부품·예산 원장(면수·수밀·슬리버·매몰면·프롭↔벨) |
| `outputs/mesh_layer2_battery_overlap_0816.json` | ⭐배터리 그룹 자기겹침 수리 — 부품 수·면·부피·질량·관성 전후 실측 |
| `outputs/mesh_cert_dimension_external_0816.json` | 바깥 참값 대조 77행 — «공식 외형이 최종 기준» 이 어디까지 맞나(§3) |
| `docs/MESH_CERTIFICATE.md` | 메쉬 인증서 — 무엇을 장담하고 무엇은 장담 못 하는가(판정 «조건부 장담») |

**한 줄 요약** — DJI 공식 제원 숫자(대각거리·외형 L×W×H·프로펠러 지름)를 담은 `DroneSpec`
데이터클래스에서 출발해, 로프트·스윕·회전체·불리언으로 드론 10종의 몸체(프레임)를 조립하고,
마지막에 공식 외형에 자동 스케일(envelope fit)해서 **치수 최악 오차 9.46%**
(DJI Mini 5 Pro), **좌우대칭 p95 ≤ 2 mm**(전 기종)를 달성하는 과정을 소스코드를 따라가며 설명한다.

⭐ **어느 판을 말하는가 — 이 편의 수는 정본 판에서 잰 값이다.** 지금 저장소의 기본값이 이것이다:

| 스위치 | 지금 기본값(정본) | 무엇이 달라지나 | 옛 판으로 되돌리는 법 |
|---|---|---|---|
| `geom.MESH_FIX_CANON` | `battery, i5` | `battery` = 배터리 팩 상자와 구조판 상자가 서로 파고든 것을 불리언 합집합으로 없앤다(4기체) · `i5` = mini2 셸의 구멍을 닫는다 | `MESH_FIX=none` |
| `geom.BLADE_LAW_CANON` | `per_airframe` | 기체마다 **그 기체의 순정 프로펠러** 평면형을 쓴다 | `BLADE_LAW=legacy` |
| 파일명 꼬리표 | `_mfixbatteryi5_blperairframe` | 정본 판 산출물의 이름에 붙는다 | 옛 판은 꼬리표가 **없다** — 그래서 두 판이 이름만으로 갈린다 |

⭐ **판정은 호출 시점에 한다.** `geom.mesh_fix_set()`·`geom.blade_law_canon()` 이 환경변수를 **부를 때마다** 읽으므로, import 뒤에 켜도 듣는다 ← 출처: `src/geom.py` 두 함수의 docstring.

⭐ **꼬리표가 규약인 이유** — 정본 판 산출물은 이름에 `_mfixbatteryi5_blperairframe` 가 붙는다. 이름이 같으면 계산기가 **옛 판 결과를 재사용**하고, 재계산이 «건너뜀» 으로 끝나 버린다 ← 출처: `benchmark/elevation_sweep_md.py` 꼬리표 블록.

⛔ `MESH_FIX=none BLADE_LAW=legacy` 를 주면 옛 판이 **비트동일**하게 다시 나온다 ← 출처: `benchmark/regress_blade_law_bitidentical.py` · `src/mesh_check.py` legacy 회귀. 전환 직전 산출물은 `/data/public/sionna/archive_pre_meshfix_20260817/` 에 있다(꼬리표 없는 샤드 3,813개 + README).

이 편의 기하·치수·대칭(§3·§4·§5)은 정본 판 원장 `report_mesh/outputs/mesh_verify_canon_0817.json`(2026-08-17 05:12 KST)에서 읽는다. 부품 높이(§2.1b·§3.2)만 노트북 생성 시점에 직접 잰다 —
그 축은 원장에 없다.

| 용어 | 뜻 |
|---|---|
| 메쉬(mesh) | 3D 표면을 작은 삼각형 조각들로 표현한 것. 전파 시뮬레이터가 먹는 형식 |
| 데이터클래스(dataclass) | '필드(변수)만 모아둔 파이썬 클래스'. 스펙 표를 코드로 옮긴 것 |
| 대각거리(diagonal, 휠베이스) | 마주보는 두 모터 축 사이 거리. 드론 크기의 대표 숫자 |
| 외형(envelope) | 암 펼침·**프로펠러 제외** 상태의 L×W×H 바운딩박스. DJI 공식 스펙 항목 |
| 바운딩박스(bounding box) | 물체를 꼭 맞게 감싸는 축 정렬 직육면체 |
| 로프트(loft) | 단면(고리)들을 순서대로 이어 붙여 곡면을 만드는 기법. 조선소에서 배 만들던 방식 |
| 초타원(superellipse) | 타원과 직사각형의 중간 곡선. 지수 n=2 면 타원, n 이 클수록 모서리 둥근 사각 |
| 스윕(sweep) | 경로(곡선)를 따라 단면을 밀어서 튜브 모양을 만드는 기법 |
| 회전체(revolve) | 옆모습 프로파일을 축 둘레로 한 바퀴 돌려 만든 형상. 도자기 물레와 같다 |
| 베지에 곡선(Bézier) | 제어점 몇 개로 정의되는 매끈한 곡선. 암의 휘어짐에 사용 |
| 불리언 합집합(boolean union) | 겹쳐 놓은 입체들을 하나로 녹여 **속에 파묻힌 면을 제거**하는 연산 |
| watertight | 구멍 없이 완전히 닫힌 표면('물을 부어도 안 새는'). 부피·법선 검증의 전제 |
| 짐벌(gimbal) | 기체가 흔들려도 카메라를 수평으로 유지해 주는 회전 마운트 |
| 챔퍼 거리(chamfer distance) | 두 점 구름 사이 '가장 가까운 짝' 거리의 통계. 대칭성 측정에 사용 |

## §0. 이 리포트의 위치 — 왜 몸체를 '직접' 만드나

mesh01~03 에서 파이프라인 전체 지도, 원본 스펙 조사, 다운로드/스캔 모델 대조를 다뤘다.
이번 편은 그 스펙 숫자가 **몸체(프레임 = 프로펠러를 제외한 모든 부위)** 메쉬가 되는 과정이다.
프로펠러(익형 블레이드)는 다음 편에서 따로 다룬다.

인터넷에서 받은 3D 모델을 쓰지 않고 **파라메트릭 CAD**(숫자를 넣으면 모양이 나오는 코드)로
직접 만드는 이유는 소스코드 머리말에 적혀 있다:

> "RCS 는 **외형(투영면적)과 재질 분포**가 결정한다. 실루엣이 틀리면 σ 가 틀린다."
> ← 출처: src/drone_cad.py 모듈 docstring

다운로드 모델은 (1) 치수가 공식 스펙과 얼마나 다른지 알 수 없고, (2) 부위별 재질 라벨(어디가
금속이고 어디가 플라스틱인지)이 없고, (3) 라이선스가 제각각이다(← 출처:
assets/meshes/reference/SOURCES.md — 다운로드 모델은 '대조용'으로만 쓴다).
파라메트릭이면 **모든 치수가 스펙에서 유도**되고, 부위마다 그룹 이름이 붙어 재질을 정확히 배정할 수 있다.

도구는 `trimesh`(메쉬 컨테이너·검증) + `manifold3d`(불리언 엔진) + `shapely`(2D 단면 폴리곤)
+ `scipy`(스플라인 보간)다. 왜 이 조합인가 — 드론 형상에는 로프트·스윕·불리언·스무딩과
watertight/법선 검증이 전부 필요한데, 이를 검증된 라이브러리에 맡기는 것이 `cadkit.py` 의
설계다(← 출처: src/cadkit.py '무엇을 쓰나·핵심 규약'). 프로젝트 자체 모듈 geom.py 는
드론 제작 도구가 아니라 **Mesh 컨테이너(꼭짓점 v·면 f·그룹 g) + 챔버·범용 프리미티브**
(box/cylinder/uv_sphere/pyramid_field 등) 담당이고, cadkit 의 `Assembly` 가 조립 결과를 마지막에
geom.Mesh 로 변환해 기존 파이프라인에 넘긴다(← src/cadkit.py to_geom docstring).

조립 흐름 한 눈에:

```
DroneSpec(공식 숫자, src/drones.py)          §1
   → build_frame_cad(부위 조립, src/drone_cad.py)   §2
   → 불리언 합집합(내부 면 제거, src/drone_cad.py)   §2
   → frame_fit_scale(공식 외형 맞춤, src/drones.py)  §3
   → 검증: C_dims(치수)·B_symmetry(대칭)                §3·§5
```

In [ ]:
# 준비 — 검증 JSON 과 스펙 원본을 읽는다 (노트북은 report_mesh/ 에서 실행된다고 가정)
import json, os, sys
sys.path.insert(0, os.path.abspath('../src'))          # 저장소 src/ — 스펙의 단일 진리원
# ⭐측정치는 **정본 판 원장**에서 읽는다(옛 원장은 아래 §2 코드 셀이 대조용으로만 쓴다).
V = json.load(open('outputs/mesh_verify_canon_0817.json', encoding='utf-8'))
VOLD = json.load(open('outputs/mesh_verify.json', encoding='utf-8'))   # 옛 판(2026-08-16)
from drones import DRONES, DroneSpec                   # ← src/drones.py (DroneSpec 정의)

print('메쉬 엔진 :', V['_meta']['mesh_engine'])
print('정본 스위치:', V['_meta']['mesh_fix'], V['_meta']['blade_law'], '· 꼬리표', V['_meta']['file_tag'])
print('검증 주파수:', V['_meta']['fc_ghz'], 'GHz')
print()
for k in V['_meta']['drones']:
    s = DRONES[k]
    print(f'{k:10s} {s.name:16s} 대각 {s.diagonal_mm:6.1f} mm · 로터 {s.num_rotors} · '
          f'프롭 {s.prop_dia_mm:5.1f} mm · {s.weight_g:.0f} g')

## §1. DroneSpec — '공식 숫자'와 '외형 파라미터'는 다른 신분이다

`DroneSpec`(src/drones.py)은 드론 한 종의 모든 정보를 담는 데이터클래스인데,
필드가 **두 부류**로 나뉜다(그리고 지금은 셋째 신분이 하나 더 있다 — 아래 (c)).
이 구분이 이 시리즈 전체의 정직성 장치다:

**(a) 실측 제원 — 웹 조사 + 독립 교차검증을 거친 숫자** (틀리면 안 되는 값)

| 필드 | 뜻 | Mini 5 Pro | Phantom 4 | 출처 |
|---|---|---|---|---|
| `diagonal_mm` | 모터-모터 대각거리 | 275 (추정, §1.1) | 350 (공식) | docs/SPECS.md·dji.com/phantom-4/info |
| `weight_g` | 이륙중량 | 249.9 | 1380 | docs/SPECS.md·dji.com/mini-5-pro/specs |
| `prop_dia_mm` | 프로펠러 지름 | 152.4 (DJI 6028F) | 240 (DJI 9450) | docs/SPECS.md·support.dji.com |
| `prop_blades`/`num_rotors` | 날개 수/로터 수 | 2/4 | 2/4 | docs/SPECS.md |
| `envelope_mm` | 공식 외형 L×W×H(프롭 제외) | —×—×91 mm — 높이만 공식 | —×—×196 mm — 이 기체도 **높이만** 강제한다 | DJI 공식(§1.1)·`src/drones.py` |

**(b) 외형 스타일 — 사진·3면도에서 눈으로 맞춘 렌더 파라미터** (실루엣 담당, 공식 스펙 아님)

| 필드 | 뜻 | Mini 5 Pro | Phantom 4 |
|---|---|---|---|
| `body_lw` | 동체 (길이,폭)/허브 비 — 접이식 슬림기는 길쭉·좁게 | (1.42, 0.66) | (1.06, 1.0) |
| `rotor_deg` | 모터 각도 배치[deg] | (56.33, 132.47, 227.53, 303.67) | (45, 135, 225, 315) (X자 기본) |
| `rotor_z_mm` | 로터별 높이 오프셋[mm] | (-7.0, 7.0, 7.0, -7.0) | 없음(전부 같은 높이) |
| `gimbal_style` | 짐벌 형태(§2) | 'single' | 'recessed' |
| `gear` | 착륙장치(§2) | 'motor_legs'(없음) | 'legs'(스키드 다리) |
| `fixed_arm` | 고정암 여부 | False (접이식) | True (고정암) |
| `body_frac` | 동체 크기/대각 비 | 0.46 | 0.52 |

← 출처: 필드 정의와 주석 src/drones.py, 값은 src/drones.py(mini5pro)·163-173(phantom4)

왜 나눴나 — (a)는 **RCS·마이크로도플러 물리에 직접 들어가는 값**이라 출처와 신뢰도(confidence)를
달고 관리하고, (b)는 실루엣(그럴듯한 겉모습)만 담당해서 사진과 눈대중으로 조정해도 되는 값이기
때문이다. 섞어 두면 '어느 숫자가 검증된 것인지' 나중에 알 수 없게 된다.

**(c) ⭐ 세 번째 신분 — 기체마다 다른 «프로펠러 평면형»**

위 두 부류만으로는 지금 상태를 다 적을 수 없다. 프로펠러 날의 평면형(최대시위 `c_max/R` 과
시위가 반경을 따라 어떻게 변하는가)이 **기체 데이터**가 됐기 때문이다 — 정본 날 법칙
`BLADE_LAW_CANON = "per_airframe"`(`src/geom.py`)에서 기체 키로 꺼내 쓴다.

- (a) 처럼 **공표 숫자**가 아니다 — 제조사가 공표하는 것은 프롭 **지름과 피치**뿐이다.
- (b) 처럼 **눈대중**도 아니다 — 기체마다 그 기체의 프롭을 실제로 잰 값이고, 근거 등급과
  불확실도가 칸마다 붙는다(공식 3D [A] 부터 근거 0 인 대리 [D] 까지).

⇒ 그래서 이 축의 출처 장부는 mesh03 §3.1.2 가, 값과 그 파급은 mesh05(프로펠러 편)가 맡는다.
이 편은 **몸체**만 다루므로 여기서는 «신분이 하나 더 있다» 는 사실까지만 적는다.
⚠ 프롭 평면형이 바뀌면 이 편의 삼각형 수도 함께 움직인다(§4) — 프롭이 기체 면수의 절반 가까이다.

← 출처: `src/geom.py` `BLADE_LAW_CANON` · `outputs/prop_law_by_airframe_0816.json`.

### §1.1 추정값은 추정값이라고 적는다 — 세 가지 사례

**사례 1: Mini 5 Pro 의 대각거리는 DJI 가 공개하지 않는다.**
스펙의 note 필드에 그대로 남겨 놨다(← 출처: src/drones.py):

> "Diagonal (250 mm) not published by DJI — was estimated from the unfolded shape."

조사 시점 추정 250 mm 이었는데, 공식 외형에 맞춘 뒤 로터 좌표(±76, ±114 mm ← 조사 근거,
src/drones.py 주석)에서 역산하면 **275 mm** 가 된다. 심지어 언폴드 L×W
조차 '프롭 제외' 값은 비공개다 — DJI 가 공개한 것은 폴디드 157×95×68 과 언폴드(**프롭 포함**)
304×380×91 뿐이라, envelope 은 **높이 91 mm 만** 강제하고 L/W 는 로터
배치가 정하게 뒀다(← 출처: src/drones.py 주석, docs/SPECS.md Mini 5 Pro 검증 절).

**사례 2: Mavic 4 Pro 는 추정 대각과 공식 외형이 서로 모순이었다.**
추정 400 mm 로는 공식 외형 328.7×390.5 mm 를 기하학적으로 만들 수 없다(대각 400 짜리 사각형은
이 외형보다 작다). 그래서 **공식 외형이 이기고**, 대각은 외형에서 유도한 441.0 mm 로
갱신했다(← 출처: src/drones.py note — "The envelope (official) wins; diagonal_mm is
kept only as an arm/motor thickness scale").

**사례 3: Phantom 4 는 대각 350 mm 가 진짜 공식이다.** 공표 외형
289.5×289.5×196 mm 도 DJI Quick Start Guide v1.2(프롭 제외)의 공식 숫자다(← 출처:
src/drones.py 주석, docs/SPECS.md Phantom 4 절·fullcompass.com 공식 스펙시트 PDF).
다만 코드가 **강제하는 축은 높이 하나**다 — `envelope_mm = —×—×196 mm`. 가로·세로까지
강제하면 그 배율이 모터 위치를 함께 밀어 **공식 대각이 도리어 틀어지기** 때문이고,
그래서 지금 이 기체의 대각 오차는 +0.00 % 다(§3.1).

이렇게 '어느 숫자가 공식이고 어느 숫자가 추정인지'를 필드 단위로 기록해 두면, §3 의 자동 맞춤이
**무엇을 기준으로 삼아야 하는지**(공식 외형 > 추정 대각)가 코드에서 결정 가능해진다.

In [ ]:
# 전 기종의 '공식 숫자' 필드 — 값은 전부 src/drones.py 의 DRONES 에서 읽는다
print(f"{'key':10s} {'이름':16s} {'대각[mm]':>8s} {'외형 L×W×H [mm]':>22s} {'프롭[mm]':>8s} {'로터':>4s} {'신뢰도':>6s}")
for k in V['_meta']['drones']:
    s = DRONES[k]
    env = ('×'.join('?' if e is None else f'{e:g}' for e in s.envelope_mm)
           if s.envelope_mm else '(없음)')
    print(f'{k:10s} {s.name:16s} {s.diagonal_mm:8.1f} {env:>22s} '
          f'{s.prop_dia_mm:8.1f} {s.num_rotors:4d} {s.confidence:>6s}')
print()
print('note(주의 문구) 첫 문장 — 추정/모순이 있으면 여기 적혀 있다:')
for k in V['_meta']['drones']:
    print(f'  {k:10s}:', DRONES[k].note.split('. ')[0][:110])

## §1b. 닮음은 어디서 오나 — 사진을 베끼지 않고 닮게 만드는 법

완성된 메쉬가 실물과 닮아 보여서 자주 받는 질문: **"3D 모델을 가져와서 스펙에 맞게 고친 건가?"** — 아니다. **어떤 외부 3D 메쉬도 기하로 가져오지 않았다.** 다만 «전부 스펙에서 나왔다» 도 정확한 말이 아니다. 닮음은 **네 층**이 쌓여 만들어지고, 층마다 근거의 성격이 다르다:

| 층 | 무엇이 | 어떻게 닮음을 만드나 | ← 출처 |
|---|---|---|---|
| **① 공표 치수** | 대각·언폴드 L×W×H·프롭 지름 | 크기·비율·로터 배치를 **숫자로 강제** — 비율이 정확하면 실루엣의 절반은 이미 맞는다. 빌드 끝에 `frame_fit_scale` 이 외형을 공식 envelope 에 맞춘다(§3) | 제조사 공식 스펙 → `docs/SPECS.md`(URL 포함) → `DroneSpec` |
| **② 형태 관찰** | 제품 사진·공식 소개에서 **사람이 뽑은 형태 특징** | "마빅=눈물방울 동체+등 배터리+렌즈 3개 짐벌", "팬텀=고정암+착륙다리", "미니=앞뒤 로터 높이 차" 같은 관찰을 **코드 파라미터로 번역**(`gimbal_style`·`gear`·`rotor_deg`·`body_lw`…) | `docs/SPECS.md` 의 기체별 '착륙장치/짐벌/색상' 항목 → `src/drone_cad.py` 기종별 분기 주석 |
| **③ 사진 계측** ⭐ | 공표 숫자가 **안 정하는** 치수(셸 비율·암 폭·다리 길이·짐벌 크기) | 사진 안에서 길이를 아는 것 하나로 mm/px 축척을 잡고 **픽셀로 잰다.** 값마다 밴드(대개 ±15 %)와 픽셀 근거를 함께 남긴다 | `assets/photos/` → `drone_cad._SHELL_SHAPE`·`_ARM_WIDTH`·`_ARM_SECTION`, `src/drones.py` note (자세히는 mesh03 §1.5) |
| **④ 실물 CAD** | 공식 CAD 가 있는 기체의 형상 상수 | matrice4e 는 공식 STEP 로 상수 14건, mini2 는 공식 GLB 로 전 상수를 **직접 실측**해 넣는다 | `outputs/meshfix_matrice4e.json` · `assets/meshes/reference/` (mesh03 §3.2) |
| **⑤ 사후 채점** | 실기체 스캔·외부 CAD 와 **사후 대조** | 만들고 나서 닮음을 측정으로 확인 — Phantom 4 스캔점의 절반이 CAD 표면 4.6 mm 이내(→ mesh08 §3) | `mesh_verify.json` G_scan · `outputs/real_cad_compare.json` |

즉 **② 가 '어떤 특징을 만들지'를 정하고, ①③④ 가 '그 특징의 크기'를 정하고, ⑤ 가 '그래서 닮았는가'를 채점**한다. ⚠ ⑤ 중에서 **제작에 한 번도 안 들어간 채점자는 실기체 스캔뿐이다** — 공식 CAD 는 ④ 로 제작에 들어갔으므로 같은 CAD 로 다시 재면 «독립 검증» 이 아니라 «반영 확인» 이다(mesh03 §3.2·mesh08 §1.5). 외부 3D 메쉬를 가져오지 않는 이유는 mesh01 §2(라이선스·치수 미검증·부위=재질 불가)에서 다뤘다.

## §2. 조립 순서 — build_frame_cad 를 소스코드 따라 걷기

프레임 조립은 `build_frame_cad(spec)`(src/drone_cad.py:2390) 한 함수가 담당한다.

⚠ **«한 가지 공통 순서» 가 아니다 — 동체를 짓는 길이 셋이다.** 아래 표는 생성 시점에 10종을
실제로 조립하면서 **어느 함수가 불렸는지 추적해** 만든 것이다(손으로 적은 목록이 아니다):

| 동체 경로 | 무엇인가 | 쓰는 기종 |
|---|---|---|
| `_body_folding` + `_canopy` (src/drone_cad.py:634) | 몰드 셸 — 초타원 단면을 길이 방향으로 로프트한 뒤 등에 캐노피를 얹는다 | Mini 5 Pro·Mavic 4 Pro·Matrice 4E·Phantom 4·Phantom 3 Professional·Mini 2 |
| `_body_plate_stack` (src/drone_cad.py:1215) | 판 스택 — 셸이 없는 열린 프레임. 카본 상·하판 + 스탠드오프 | S1000+·X500 V2 |
| `_body_profiled` (src/drone_cad.py:1776) | 평면형 기반 — x 가 아니라 **z 를 따라** 단면을 쌓아 위에서 본 윤곽을 먼저 정한다 | Typhoon H (H480)·Matrice 350 RTK |

암도 둘로 갈린다 — 접이식 스윕 `_arm_folding`(Mini 5 Pro·Mavic 4 Pro·Matrice 4E·Phantom 4·Phantom 3 Professional·Mini 2) · 카본 튜브 `_arm_tube`(S1000+·X500 V2) · 상반각 튜브 `_arm_dihedral`(Typhoon H (H480)·Matrice 350 RTK).

아래 1~4단계는 **몰드 셸 경로**(가장 많은 기종이 쓰는 길)를 따라 걷는다.

**1단계. 동체 — 초타원 단면의 로프트** (`_body_folding`, src/drone_cad.py:634)

동체 길이 방향(x)의 6개 지점마다 반폭·반높이·중심높이를 정해 두고(코는 좁고 살짝 처지고,
허리에서 가장 넓고, 꼬리는 완만히 좁아진다), 스플라인으로 30개 단면으로 보간한 뒤
(`spline_sections`, src/cadkit.py:325) 이어 붙인다(`loft`, src/cadkit.py:276).

**왜 단면이 원이 아니라 초타원인가** — 실제 드론 동체 단면은 순수 타원도 상자도 아니고 그 중간,
'모서리가 둥근 각진 타원'이다. 초타원의 지수 n 하나로 이 정도를 조절한다(← 출처:
src/cadkit.py:307 docstring — "실제 드론 단면은 순수 타원도 박스도 아니고 이 중간이다").
지수 n 과 코 처짐은 **기종별 형상표**(`drone_cad._SHELL_SHAPE`)가 정한다 — 지금 값은:

| 기체 | 초타원 지수 n | 코 처짐 |
|---|---|---|
| Mini 5 Pro | 3.2 | 0.18 |
| Mavic 4 Pro | 3.1 | 0.14 |
| Matrice 4E | 3.2 | 0 |
| Phantom 4 | 3.4 | 0.09 |
| Phantom 3 Professional | 3 | 0.1 |
| Mini 2 | 3.6 | 0 |

n 이 클수록 단면이 상자에 가깝고(모서리가 각지고), 코 처짐이 0 이면 기수가 안 숙는다.
이 값들은 사진·CAD 대조로 기종마다 따로 잡혔다 — 손으로 고른 «접이식 공통값» 이 아니다.

**2단계. 캐노피 — 등에 얹힌 배터리 돔** (`_canopy`, src/drone_cad.py:1204)

실물은 배터리가 동체 등에 얹힌 낮고 평평한 돔 모양이라, 동체보다 작은 로프트를 하나 더 만들어
z 위로 올려 붙인다. 그룹 이름은 `canopy` — 재질 배정(플라스틱)이 그룹 단위로 따라온다.

**3단계. 암 — 베지에 경로의 스윕** (`_arm_folding`, src/drone_cad.py:1301)

허브(동체 가장자리)에서 모터 위치까지 2차 베지에 곡선으로 **완만히 위로 휘는** 경로를 만들고,
그 경로를 따라 '둥근 직사각' 단면을 밀어(스윕) 테이퍼 튜브를 만든다. 원기둥을 꽂는 대신 스윕을
쓰는 이유: 실물 접이식 암은 직선 봉이 아니라 동체에서 모터로 갈수록 가늘어지며 휘는 형상이기
때문이다. 굵기는 대각거리에 비례시키되 고정암(Phantom)은 더 굵게 — arm_r0 = 접이식 0.055·diag
vs 고정암 0.085·diag. ⚠ 실측 폭이 있는 기종은 그 비례식 대신 **사진에서 잰 뿌리·끝 폭**을 쓴다(`drone_cad._ARM_WIDTH`).

**4단계. 모터 벨 — 회전체** (`_motor_bell`, src/drone_cad.py:132)

(r,z) 프로파일 9개 점을 z축으로 한 바퀴 돌린 회전체다. "아래가 잘록하고 위가 부푼 실제
아웃러너(outrunner: 겉통이 도는 드론 모터) 형상"(← 해당 함수 docstring). 회전체 구현에는
규약이 하나 있다 — **r=0 인 점은 링이 아니라 하나의 꼭짓점(apex)으로 접는다**. 링으로 두면
같은 자리에 정점이 seg개 생겨 면적 0 퇴화 삼각형이 쏟아지고, 퇴화면은 법선이 정의되지 않아
PO/SBR 의 조명판정(n̂·û>0)을 오염시키기 때문이다(← 출처: `revolve` src/cadkit.py:386 주석).

### §2.1 짐벌·착륙장치 분기 — 드론마다 왜 다르게 만들었나

짐벌과 착륙장치는 **드론 실루엣의 핵심 식별 특징**이라 기종별로 함수를 나눴다.
착륙장치는 스펙 필드 `DroneSpec.gear` 가 분기를 정하고, **짐벌은 아직 기종 키로 갈린다.**

⭐ 아래 두 표의 «쓰는 기종» 열은 **추적한 값**이다 — 생성 시점에 10종을 실제로 조립하면서
어느 함수가 불렸는지 기록했다. 줄 번호도 `inspect` 로 읽는다.

**짐벌·카메라**

| 함수 | 형태 | 쓰는 기종 | 왜 |
|---|---|---|---|
| `_gimbal_hasselblad` (1364행) | 넓은 3렌즈 블록 + 요크 | Mavic 4 Pro | 실물이 기수와 일직선인 **큰 전면 짐벌**(3렌즈)이라서 — 매달린 상자로 만들면 실루엣이 틀린다 |
| `_gimbal_sensor_v2` (1470행) | 3축 마운트 + 렌즈 3 + 레이저 측거 | Matrice 4E | 측량 페이로드(카메라 클러스터 + 레이저 거리계)가 공식 구성이라서 — RTK 돔도 캐노피 위에 추가된다 |
| `_gimbal_compact3` (1397행) | 좌우로 넓은 소형 3축 유닛 | Mini 5 Pro·Mini 2 | Mini 계열은 기수 함몰부에 들어앉은 컴팩트 유닛이라 Phantom 류의 방진판·요 샤프트가 없다 |
| `_gimbal_hanging` (1447행) | 방진판 + 요크 + 카메라 상자 + 렌즈 | S1000+·Phantom 4 | 코 아래 **매달린** 전통 짐벌. Phantom 은 함몰(recessed)이라 동체에 더 붙여 배치 |
| `_gimbal_cgo3` (1859행) | CGO3 실물 부품 치수 그대로 | Typhoon H (H480) | 실물 CAD 가 있는 기체라 부품 치수를 그대로 세운다 |
| (전용 조립) | 방진판 + 요 샤프트 + 카메라 블록 | Phantom 3 Professional | 매뉴얼 정투영 정면도·측면도에서 부품마다 따로 실측한 값을 쓴다 — 공용 함수의 내부 비율과 안 맞았다 |
| (짐벌 없음) | — | X500 V2·Matrice 350 RTK | 카메라가 아예 없는 개발 프레임. `gimbal_style='none'` 으로 **선언**한다 — 선언이 없으면 조립이 예외를 던진다 |

⚠ `_gimbal_infinity` 는 소스에 남아 있지만 **지금 함대에서 부르는 기체가 없다**(추적 결과 0종).
코드에 있는 함수 목록을 그대로 옮겨 적으면 «Mavic 4 Pro 가 쓴다» 처럼 틀리게 된다.

**착륙장치**

| 함수 | 형태 | 쓰는 기종 | 왜 |
|---|---|---|---|
| `_gear_arch` (1630행) | 뒤집힌 U 자 아치 다리 2개 | Phantom 4·Phantom 3 Professional | 일체형 흰 셸에 붙은 고정 착륙다리가 Phantom 정체성. 비율은 **정면사진 실측** |
| `_gear_tall_tube` (1881행) | 길게 벌어지는 카본 봉 + 스키드 바 | S1000+·Typhoon H (H480)·X500 V2 | 벨리 짐벌·페이로드 공간을 확보하는 긴 다리 |
| `_gear_arm_spikes` (1672행) | 암 끝 아래로 뻗은 짧은 발 4개 | Matrice 4E | 전용 스키드 없이 **암 끝**으로 앉는 기종 |
| `_gear_motor_legs` (2229행) | 앞 모터 포드 밑 두 갈래 다리 2개 | Mini 5 Pro·Mavic 4 Pro·Mini 2 | 접이식 소형기는 전용 스키드 대신 **앞쪽 모터 포드 아래에만** 다리가 달린다(뒤쪽 포드는 매끈하다) — 제품사진 계측, 밴드 ±15 % |
| (인라인 조립) | 카본 튜브 다리 + 스키드 | Matrice 350 RTK | 다리 형상이 기종 전용이라 자기 분기 안에서 직접 스윕한다 |

⚠ `_gear_skids`(1726행) · `_gear_tall`(1739행) · `_gear_feet`(2220행)도 소스에 있지만 지금 부르는 기체가 없다 — 앞의 두 함수를 쓰는 자리를 각각 `_gear_arch` · `_gear_tall_tube` 가 받았고, `_gear_feet` 는 자기 docstring 이 스스로 **폐기된 옛 형상**이라고 적는다.

⭐ 이 표가 다시 낡지 않게 하는 방법이 요점이다 — **소스를 읽어 옮겨 적지 말고, 돌려서 기록한다.**
기종을 추가하면 표가 저절로 따라온다.

### §2.1b ⚠ 짐벌 조립의 지금 남은 어긋남

짐벌은 «작은 부품» 처럼 보이지만 이 함대에서 가장 무거운 결함이 여기 있다. 전용 검사기가
게이트 넷(A 부착 · B 뜸/삼킴 · C 선언 대비 실제 크기 · D 재질 민감도)으로 10종을 훑는다
← 출처: `outputs/mesh_inspect_gimbal_sensors_0816.json` · `benchmark/check_gimbal_sensors_0816.py`.

**① 게이트 A — 무엇이 기체의 «바닥» 인가.** 이 게이트가 보는 것은 «카메라가 착륙발보다 아래로
내려와 기체의 최저점이 되는가» 다. 최저점은 세로 배율을 정하므로(§3.2) 카메라 하나가 기체 전
부위의 세로 치수를 흔들 수 있다. 지금 함대에서는 **카메라가 최저점인 기체가 없다** —
착륙발이 있는 기체의 여유는 이만큼이다(생성 시점 실측, (+)면 카메라가 발보다 위):

| 기체 | 카메라 최저점 − 착륙발 최저점 |
|---|---|
| DJI Mini 5 Pro | +12.64 mm |
| DJI Mavic 4 Pro | +3.41 mm |
| DJI Matrice 4E | +20.08 mm |
| DJI S1000+ | +201.61 mm |
| DJI Phantom 4 | +83.28 mm |
| Yuneec Typhoon H (H480) | +20.34 mm |
| DJI Phantom 3 Professional | +25.70 mm |
| DJI Matrice 350 RTK | +174.38 mm |
| DJI Mini 2 | +7.97 mm |

**② 게이트 C — 헬퍼에 넣은 «선언 치수» 와 실제로 지어지는 크기가 다르다.** 헬퍼가 인자로 받은
상자 크기 위에 요크·렌즈·마운트를 더 붙이기 때문이고, 지금 그 배율은 이만큼이다:

| 헬퍼(인자) | 쓰는 기종 | 선언 [mm] | 실제 [mm] | 배율 |
|---|---|---|---|---|
| `_gimbal_compact3(50,33,30)` | mini5pro | 30×50×33 | 43.5×50×46.04 | 1.45×1.00×1.40 |
| `_gimbal_compact3(32.24,24.38,27.98)` | mini2 (역산 인자 — built 가 공식 CAD bbox 40.57×32.24×34.01 과 일치) | 27.98×32.24×24.38 | 40.57×32.24×34.01 | 1.45×1.00×1.40 |
| `_gimbal_sensor_v2(59,61.2,52)` | matrice4e | 52×59×61.2 | 60.84×80.83×78.95 | 1.17×1.37×1.29 |
| `_gimbal_hasselblad(s=50)` | mavic4pro | 81.7×95×95 | 95.48×100×95 | 1.17×1.05×1.00 |
| `_gimbal_hanging(52,48,56)` | phantom4 | 56×52×48 | 85.2×78×56.64 | 1.52×1.50×1.18 |
| `_gimbal_hanging(100,75,100)` | s1000plus | 100×100×75 | 157.5×150×88.5 | 1.57×1.50×1.18 |

⭐ 읽는 법: **인자를 실물 치수라고 믿고 쓰면 안 된다.** 실물 치수를 맞추려면 mini2 처럼 «원하는 결과 bbox → 인자» 로 **역산**해야 한다(그 기체는 그렇게 해서 공식 CAD bbox 와 맞는다).

**③ 게이트 B — 짐벌 부품이 기체에 안 닿거나(뜸) 다른 부품에 완전히 묻힌다(삼킴).** 뜬 것은 Phantom 3 Professional, 삼켜진 것은 Phantom 4, Phantom 3 Professional 에 있다. σ 로는 작지만 **PO 와 SBR 이 같은 메쉬를 다르게 읽게** 만든다 — 묻힌 면을 SBR 은 가리고 PO 는 센다(mesh07 §8.3).

### §2.2 보이지 않는 부품 — 내부 battery/pcb 를 왜 넣나

조립 마지막에 렌더에선 절대 안 보이는 상자 두 개가 셸 **안에** 들어간다(← `src/drone_cad.py` 내부 산란체 주석):

> "내부 금속 산란체 (RCS 지배) — 셸 안이라 렌더엔 안 보이지만 PO/SBR 이 센다"

이유: 드론 셸은 플라스틱이라 GHz 전파에 **반투명**하다(진폭 반사계수 |Γ|≈0.24~0.28 — 즉 전파
대부분이 셸을 뚫고 들어간다 ← 출처: src/materials.py plastic 정의·note). 그래서 실물
드론의 레이더 반사는 셸이 아니라 **안에 있는 배터리팩·ESC/메인보드·모터 금속**이 지배한다
(← 출처: `src/drone_cad.py` "내부 금속 산란체 (RCS 지배)" 주석). 재질 배정도 이에 맞춰
battery=metal("GHz 에서 파우치 포일은 사실상 금속"), pcb=FR-4+구리 그라운드플레인이다
(← src/drones.py, DRONE_GROUP_MAT). 치수는 기종별 실물이 비공개라 동체 대비 대표
비율(파라메트릭)로 정의된다(← 같은 자리, bl·bw·bh 비율) — 추정값이라는 점은
'현재 한계'로 정리에 적는다.

**⭐ 배터리 그룹은 지금 «한 덩어리» 다.** 이 그룹에는 상자가 둘 들어간다 — 팩 상자와 구조판
상자다. 정본 수리 `battery`(`MESH_FIX_CANON` 에 들어 있어 **기본으로 켜져 있다**)가 그 둘을
**불리언 합집합**으로 한 껍질로 만든다. 치수는 한 mm 도 안 바꿨다 — 상자 크기·위치는 그대로다.

| 무엇 | 수리 전 | 지금(정본) |
|---|---|---|
| `battery` 그룹의 부품 수 | 2개 | **1개** |
| 그 그룹의 삼각형 | 24장 | **44장** (합집합이 교차선에서 면을 나눈다) |
| 두 상자가 서로 파고든 비율 | 47.3~50.0 % | **0.0 %** |
| 겹쳐서 두 번 세던 면적 | 68~194 cm² | **0** |

해당 기체는 4종이다 — DJI Mini 5 Pro·DJI Mavic 4 Pro·DJI Phantom 4·DJI Mini 2. 나머지 기체는 애초에 배터리 부품이 하나이거나(겹칠 것이 없다) 기종별 실측표를 받아 이미 안 겹쳐 있었다.

**왜 이 절에 적나 — 부피·질량·무게중심이 여기서 정해지기 때문이다.** 파고든 만큼은 «두 번 센 부피» 였으므로, 겹침이 사라지면 배터리 부피가 **12.27 %** 줄고 그 사슬이 질량·관성으로 내려간다:

| 기체 | 팩 질량(밀도표 기준) | 관성 대각 | 질량중심 이동 |
|---|---|---|---|
| DJI Mini 5 Pro | 52.9 → 47.7 g (-5.3) | +1.9~+2.4 % | 0.46 mm |
| DJI Mavic 4 Pro | 214.2 → 192.7 g (-21.5) | +2.1~+2.4 % | 0.44 mm |
| DJI Phantom 4 | 319.9 → 288.8 g (-31.0) | +2.1~+2.3 % | 0.42 mm |
| DJI Mini 2 | 68.7 → 62.3 g (-6.3) | +2.2~+2.8 % | 0.26 mm |

관성 대각은 +1.9~+2.8 %, 질량중심은 0.26~0.46 mm 움직인다. **자세 응답(로터 요동) 모델이 이 값을 쓰므로**, 그쪽을 다시 잴 때는 여기 표를 보고 쓸 것.

⚠ 질량은 **밀도표에서 나온 수**이지 실측이 아니다 — 방향(줄어든다)은 확실하고 크기의 참값은 모른다. 전파 쪽 몫(PO 가 면적을 두 번 세지 않게 되는 것)의 크기는 mesh07~08 이 다룬다 (방위평균 +0.32~+2.83 dB).

⛔ **상자를 «안 겹치게 옮기지» 않은 이유** — 기종별 팩·구조판의 실측 치수가 없다. 없는 치수로
상자를 옮기면 그것은 수리가 아니라 임의로 돌린 손잡이가 된다. 그래서 **치수는 그대로 두고
겹침만 없앴다** ← 출처: `outputs/mesh_layer2_battery_overlap_0816.json` `안_고친_것과_이유`.

### §2.3 마지막 손질 — 불리언 합집합

부위를 겹쳐 쌓기만 하면 '동체 속에 파묻힌 암 뿌리' 같은 **내부 면**이 메쉬에 그대로 남아,
레이더 계산(PO/SBR)이 존재하지 않는 면을 헛세게 된다. 그래서 불리언 합집합을 돌려
겹친 파트를 한 껍질로 녹인다 — "그런 면이 **애초에 존재하지 않는다**"(← src/drone_cad.py).

⭐ **합집합이 필요한 자리가 둘이라는 것이 요점이다:**

| 어디 | 무엇이 겹치나 | 지금 |
|---|---|---|
| 그룹 **사이** | 동체 ↔ 암 뿌리, 셸 ↔ 캐노피처럼 서로 다른 부위가 만나는 자리 | 조립 규약대로 그룹별 합집합을 돈다 |
| 그룹 **안** | 같은 그룹의 두 부품이 서로 파묻히는 자리 — `battery` 의 팩 상자 ↔ 구조판(§2.2) | 정본 수리로 합쳐져 **0 %** |

둘을 나눠 적는 이유는 검사 방식이 다르기 때문이다 — 그룹 사이는 «껍질이 하나인가» 로 보고,
그룹 안은 «그 그룹 표면적 중 남의 솔리드 안에 든 비율» 로 본다. 이 겹침 검증 결과(F_overlap)와
예산표는 mesh07 에서 다룬다.

### §2.4 매끈하게 만드는 두 단계에는 대가가 있다

동체를 실물처럼 만드는 두 도구 — **스플라인 보간**(단면 사이를 잇는다)과 **Taubin 스무딩**
(각진 로프트를 다듬는다) — 은 둘 다 형상을 **바꾼다**. 지금 그 크기를 재 놓았다.

**① 스플라인이 제어점 사이에서 넘친다.** 3차 스플라인은 기울기까지 연속으로 잇느라
제어점 사이에서 값을 살짝 벗어난다. 형상표가 «잘록한 허리 → 넓은 어깨» 로 꺾이면 특히 그렇다:

| 기체 | 무엇 | 형상표 의도 | 메쉬 실측 | 어긋남 |
|---|---|---|---|---|
| matrice4e | 셸 배(아래쪽) | −30.38 mm (형상표) · −30.81 mm (공식 CAD) | −34.74 mm | **CAD 대비 3.93 mm 더 아래** |
| mini5pro | 셸 최대 반폭 | 35.21 mm | 37.87 mm | **+7.6 %** |
| mini2 | 셸 높이 | 44.80 mm | 48.36 mm | **+8.0 %** |

⭐ **부호가 늘 같은 쪽이다** — 항상 «더 크게(= 전파에 더 밝게)». 우연이 아니라 넘침의 성질이다.
고치는 길은 제어점을 6 → 8~10 으로 늘리거나, 넘치지 않는 보간(단조 스플라인)으로 바꾸는 것이다.
둘 다 전 기종 메쉬가 바뀌므로 별도 라운드의 일이다.

**② 스무딩이 끝단 캡을 안으로 당긴다.** 로프트의 앞뒤 마감면은 삼각형 팬으로 닫혀 있는데,
스무딩이 그 테두리 링을 안쪽으로 끌어당긴다(스무딩 0회 ↔ 4회 대조):

| 기체 | 어디 | 스무딩 전 | 스무딩 후 | 남은 비율 |
|---|---|---|---|---|
| matrice4e | 기수 단면(반폭×반높이) | 41.02×36.88 mm | 23.42×21.31 mm | 약 57 % |
| mavic4pro | 꼬리 단면 | 42.63×28.57 mm | 24.58×16.43 mm | 약 57 % |
| phantom3 | 기수 단면 | 26.04×22.01 mm | 16.30×12.96 mm | 약 61 % |

**가운데 4개 스테이션은 형상표를 0.5 % 안에서 재현한다** — 손실은 끝단 전용이다.
방위평균 투영면적으로는 −0.00~−0.06 dB 밖에 안 움직이므로 **레벨 결함이 아니다.**
흔들리는 것은 «기수를 정면으로 봤을 때의 정반사 형상» 이고, 그 크기는 아직 커널로 안 쟀다.

← 출처: `outputs/mesh_inspect_body_arms_0816.json` `findings`(로프트 끝단 캡·스플라인 넘침).
고칠 자리는 선택 인자로 뚫려 있다(`_body_folding(..., smooth_iters=)`) — **기본값이 옛 값이라
지금 메쉬는 비트동일**하다.

In [ ]:
# 스펙의 스타일 필드가 실제 메쉬 그룹으로 이어졌는지 — 정본 판 원장(A_geometry)과 대조.
#  ⚠ 옛 판(VOLD)과 나란히 찍는다 — 정본 전환으로 프롭 면수가 달라져 총 삼각형 수가 바뀐다.
print(f"{'key':10s} {'gimbal_style':>13s} {'gear':>5s} {'암':>10s}   옛tri → 정본tri · 그룹")
for k in V['_meta']['drones']:
    s = DRONES[k]; g = V['A_geometry'][k]; g0 = VOLD['A_geometry'][k]
    arm = '고정암' if s.fixed_arm else '접이식'
    same = '=' if g0['n_faces'] == g['n_faces'] else '→'
    print(f"{k:10s} {s.gimbal_style:>13s} {s.gear:>5s} {arm:>8s}   "
          f"{g0['n_faces']:,} {same} {g['n_faces']:,} · {sorted(g['groups'])}")
# gear 필드가 어떤 값이든 다리를 짓는 기종에는 'gear' 그룹이 있어야 정상
#  (mini5pro·mavic4pro 는 gear='motor_legs' — 앞 모터 포드 밑 다리를 짓는다)

## §3. envelope fit — 공식 외형에 자동으로 맞추기 (frame_fit_scale)

실루엣 파라미터를 눈으로 아무리 다듬어도, 완성된 프레임의 바운딩박스가 DJI 공식 L×W×H 와
같아진다는 보장이 없다 — 파라메트릭 비율(body_frac·body_lw 등)은 실루엣용이지 치수 보증용이
아니기 때문이다.

이게 왜 중요한가 — 챔버 기하(낮은 앙각 el≈15°)에서는 **높이가 측면 투영면적을 지배**하고,
평판 극한에서 RCS 는 σ ∝ (투영면적)² 이므로, 높이가 수십 % 어긋나면 σ 가 수 dB 단위로
틀어진다. 즉 모양이 예뻐도 크기가 틀리면 탐지 확률 계산이 통째로 틀린다(← 출처:
src/drones.py envelope fit 절 주석).

**규약** (`frame_fit_scale`, src/drones.py): 실루엣은 그대로 두고, 완성된 프레임의
바운딩박스를 재서 **공식 envelope_mm 과 같아지도록 축별 배율 (sx, sy, sz)** 를 곱한다.
공식값이 없는 축(None)은 건드리지 않는다 — Mini 5 Pro 는 높이만 맞추는 이유가 이것이다(§1.1).
모터 위치(`rotor_layout`)에도 같은 배율을 걸어 프로펠러가 모터 위에 정확히 앉는다
(← src/drones.py — "프레임과 **같은** 외형보정 배율").

**대가도 명시돼 있다**(← src/drones.py): 축마다 배율이 다르므로(비등방)
모터 원통이 약간 타원이 된다. "RCS 가 보는 것은 투영면적과 외형이므로 이쪽을 맞추는 것이
옳다는 판단." 그리고 **프로펠러는 스케일하지 않는다** — 프롭 지름(prop_dia_mm)은 별도 공식
스펙이 있기 때문이다(← src/drones.py).

### §3.1 결과 — 치수 대조 (공식 → 실측, 오차%)

아래 수치는 완성 메쉬를 실제로 재서 공식값과 비교한 것이다 — **정본 판**(`MESH_FIX=battery,i5` · `BLADE_LAW=per_airframe`)에서 이 노트북을 만들 때 다시 쟀고,
자는 검증 스위트가 쓰는 함수 그대로다(← `src/drones.py` `frame_envelope_mm` ·
`report_mesh/src/verify_mesh_suite.py` `sec_C_dims`):

| 기종 | L [mm] | W [mm] | H [mm] | 대각 [mm] | 프롭 [mm] | 최악오차 | fit_scale (sx,sy,sz) |
|---|---|---|---|---|---|---|---|
| DJI Mini 5 Pro | — (공식값 없음) | — (공식값 없음) | 91.0 → 91.0 (-0.00%) | 275.0 → 249.0 (-9.46%) | 152.4 → 152.4 (-0.00%) | **9.46%** | (1.000, 1.000, 1.299) |
| DJI Mavic 4 Pro | — (공식값 없음) | — (공식값 없음) | 135.2 → 135.2 (+0.00%) | 441.0 → 441.0 (+0.00%) | 267.0 → 267.0 (-0.00%) | **0.00%** | (1.000, 1.000, 1.598) |
| DJI Matrice 4E | — (공식값 없음) | — (공식값 없음) | 149.5 → 149.5 (+0.00%) | 438.8 → 439.1 (+0.08%) | 274.0 → 274.0 (-0.00%) | **0.08%** | (1.000, 1.000, 1.000) |
| DJI S1000+ | 1016.0 → 1016.0 (+0.00%) | 1016.0 → 1016.0 (+0.00%) | 380.0 → 380.0 (+0.00%) | 1045.0 → 1043.5 (-0.14%) | 381.0 → 381.0 (-0.00%) | **0.14%** | (0.999, 0.999, 0.929) |
| DJI Phantom 4 | — (공식값 없음) | — (공식값 없음) | 196.0 → 196.0 (+0.00%) | 350.0 → 350.0 (+0.00%) | 240.0 → 240.0 (-0.00%) | **0.00%** | (1.000, 1.000, 0.937) |
| Yuneec Typhoon H (H480) | — (공식값 없음) | — (공식값 없음) | 310.0 → 310.0 (+0.00%) | 480.0 → 480.0 (+0.00%) | 230.2 → 230.2 (-0.00%) | **0.00%** | (1.000, 1.000, 1.019) |
| Holybro X500 V2 | — (공식값 없음) | — (공식값 없음) | — (공식값 없음) | 500.0 → 500.0 (+0.00%) | 254.0 → 254.0 (-0.00%) | **0.00%** | (1.000, 1.000, 1.000) |
| DJI Phantom 3 Professional | — (공식값 없음) | — (공식값 없음) | 185.0 → 185.0 (+0.00%) | 350.0 → 350.0 (+0.00%) | 240.0 → 240.0 (-0.00%) | **0.00%** | (1.000, 1.000, 0.998) |
| DJI Matrice 350 RTK | — (공식값 없음) | — (공식값 없음) | 430.0 → 430.0 (+0.00%) | 895.0 → 895.0 (+0.00%) | 533.4 → 533.4 (-0.00%) | **0.00%** | (1.000, 1.000, 0.991) |
| DJI Mini 2 | — (공식값 없음) | — (공식값 없음) | 56.0 → 56.0 (-0.00%) | 213.0 → 215.3 (+1.10%) | 119.1 → 119.1 (-0.00%) | **1.10%** | (1.000, 1.000, 1.001) |

읽는 법 — envelope 을 직접 맞춘 L/W/H 는 오차 0%. 전체 최악은 DJI Mini 5 Pro 의 대각
-9.46% 다(공식 275.0 → 실측 249.0).
⚠ 이 기체의 대각 오차는 **배율 탓이 아니다** — 면내 배율이 1.000 이라 늘리거나 줄인 것이 없다.
원인은 로터를 사다리꼴로 놓은 **선언된 선택**이고(§5), 그래서 마주보는 로터 거리가 공표 대각보다
짧게 나온다.

**가로·세로를 강제하는 기체는 지금 1종뿐이다.** 나머지는 **높이만** 강제한다 — 면내 배율이 1.000 이면 모터 위치가 안 밀리고, 그래서 공표 대각이
그대로 남는다(예: DJI Phantom 4 대각 +0.00 % · DJI Matrice 4E +0.08 %).
⭐ 읽는 규칙 하나 — **오차 0 인 축은 «검증» 이 아니라 «구성상 보장»** 이다. 그 축은 배율로 맞춘
것이라 언제나 0 이 나온다. 진짜 정보는 «강제하지 않은 축이 얼마나 맞는가» 쪽에 있다.

프로펠러 지름 오차가 전 기종 -0.00~-0.00% 로 좁게 모이는 것도 같은 성질이다 — 빌드가 **날을 지은 뒤
실제 최대반경을 재서 공표 지름으로 되돌리기** 때문이다. «맞다» 가 아니라 «맞춰 놓았다» 로 읽어야
한다(mesh05 §6).

fit_scale 을 보면 기종마다 실루엣이 공식 외형에서 얼마나 멀었는지도 보인다:
DJI Mavic 4 Pro 는 (1.000, 1.000, 1.598), S1000+ 는 수평
(0.999, 0.999) 다.

### §3.1b ⚠ «공식 외형이 최종 기준» 이 어디까지 맞나 — 바깥 참값

이 절의 논지는 «공표 외형을 기준으로 삼는다» 인데, 그 기준이 닿지 않는 자리가 있다.
공표 외형은 기체 하나를 감싸는 상자 **한두 줄**뿐이라, 부품 하나하나의 크기는 구속하지 않는다.
저장소 **밖의 참값**과 부품 단위로 견주는 표가 따로 있고(`src/mesh_dimref.py::REFS`, 77행),
지금 그중 **24행이 어긋나 있다**. 큰 것부터:

| 기체 | 무엇 | 참값 | 잰 값 | 오차 |
|---|---|---|---|---|
| DJI Mini 5 Pro | 배터리 높이(`battery`) | 24.85 | 32.17 | **+29.5 %** |
| DJI Mini 5 Pro | 배터리 길이(`battery`) | 86.10 | 68.98 | **-19.9 %** |
| DJI Mini 5 Pro | 배터리 폭(`battery`) | 54.89 | 45.95 | **-16.3 %** |
| DJI Matrice 4E | 셸 배(아래) z(`body`) | -30.81 | -34.74 | **-12.8 %** |
| DJI Matrice 350 RTK | 펼침 폭(프롭 제외)(`airframe`) | 670.00 | 626.34 | **-6.5 %** |
| DJI Phantom 3 Professional | 높이(모터 상단 → 발)(`airframe`) | 185.00 | 175.26 | **-5.3 %** |
| DJI Matrice 350 RTK | 펼침 길이(프롭 제외)(`airframe`) | 810.00 | 771.05 | **-4.8 %** |

← 출처: `outputs/mesh_cert_dimension_external_0816.json` `findings` · `docs/MESH_CERTIFICATE.md` §3.3.

⭐ 읽는 법 — 어긋남은 **두 종류**다.

- **부품 치수**(위 표의 `battery`·`body` 행) — envelope fit 이 애초에 손대지 않는 축이다. 공표 외형은 기체를 감싸는 상자만 정하고 그 안의 부품 크기는 말하지 않는다. 내부 부품은 §2.2 대로 동체 대비 비율로 세운 값이라, 실물 참값이 생기면 그때 갈아 끼울 자리다.
- **기체 외형**(`airframe` 행 — DJI Matrice 350 RTK 펼침 폭·길이, DJI Phantom 3 Professional 높이) — 높이만 강제하는 기체라 가로·세로는 «지어진 결과» 이고, 그 결과가 공표와 몇 % 벌어져 있다.

⛔ **독립 참값이 한 줄도 없는 기체가 둘**(DJI S1000+ · DJI Matrice 350 RTK)이라는 것도
함께 읽어야 한다 — 그 둘의 «치수가 맞다» 는 우리 수를 우리가 다시 읽은 것이다.

### §3.2 ⭐ 배율이 커지면 그것은 «맞춤» 이 아니라 «늘리기» 다

위 표의 `fit_scale` 은 **형상을 통째로 늘리거나 줄이는 배율**이다. 값이 1 에 가까우면
«형상이 이미 공표 외형과 맞았다» 는 뜻이라 교정이 무해하다. 그러나 값이 1 에서 크게 벗어나면,
그것은 **형상이 틀렸다는 신호를 배율로 덮은 것**이다 — 부품 하나하나가 다 함께 늘어난다.

지금 상태를 그대로 적는다:

| 기체 | 세로 배율 | 무엇을 뜻하나 |
|---|---|---|
| DJI Mini 5 Pro | 1.2985 | **형상이 세로로 부족해 전 부품을 늘려 공표 높이를 맞춘다** |
| DJI Mavic 4 Pro | 1.5977 | **형상이 세로로 부족해 전 부품을 늘려 공표 높이를 맞춘다** |
| DJI Matrice 4E | 0.9999 | 형상이 이미 맞는다(교정 무해) |
| DJI S1000+ | 0.9291 | 형상이 세로로 넘쳐 눌러 맞춘다 |
| DJI Phantom 4 | 0.9373 | 형상이 세로로 넘쳐 눌러 맞춘다 |
| Yuneec Typhoon H (H480) | 1.0189 | 형상이 이미 맞는다(교정 무해) |
| Holybro X500 V2 | 1.0000 | 형상이 이미 맞는다(교정 무해) |
| DJI Phantom 3 Professional | 0.9981 | 형상이 이미 맞는다(교정 무해) |
| DJI Matrice 350 RTK | 0.9908 | 형상이 이미 맞는다(교정 무해) |
| DJI Mini 2 | 1.0013 | 형상이 이미 맞는다(교정 무해) |

← 출처: `report_mesh/outputs/mesh_verify_canon_0817.json` `C_dims.*.fit_scale`(정본 판).

**DJI Mini 5 Pro 와 DJI Mavic 4 Pro 가 그 자리다.** 형상표(사진 계측)의 셸 높이
45.05 / 62.10 mm 가 지어진 메쉬에서는
60.00 / 103.61 mm 로 나온다 — 셸만 그런 것이 아니라 암·모터·짐벌까지 **전 부품이 같은 배율로** 늘어난다.

⚠ DJI Mavic 4 Pro 의 세로 배율이 이 함대에서 가장 크다(1.5977). 이 기체는 **착륙발이 최저점**이고(카메라 최저점이 발보다 3.41 mm **위**다), 그래서 공표 높이와 형상의 차이가 전부 세로 배율로 드러난다 — 가려 주는 것이 없다.

⚠ 이 표의 배율은 **형상 결함의 크기**이지 σ 가 아니다. σ 로 얼마인지는 커널이 답할 문제이고,
방위평균 상한은 mesh07~08 에서 다룬다.

⚠ **아직 못 정한 것** — 공표 언폴드 높이는 공식이지만, 그 높이를 다리·셸·짐벌·모터에 **어떻게**
나누는지는 사진 한 장으로 안 풀린다. matrice4e 처럼 공식 CAD 가 필요한데 DJI 는 Mavic 4 Pro
CAD 를 공개하지 않는다 ← 출처: `outputs/mesh_inspect_body_arms_0816.json` `not_settled_this_round`.

**대조군** — 공식 CAD 로 세로 배분을 다시 푼 matrice4e 는 배율이
0.9999 로, 형상 자체가 공표 높이를 낸다.

## §4. 드론별 개성 — 같은 코드, 다른 스펙, 다른 드론

조립 코드는 하나지만 스펙 필드가 다르니 다른 드론이 나온다. 배정된 두 그림으로 양 극단을
대비한다: **접이식 초소형**(Mini 5 Pro)과 **고정암 클래식**(Phantom 4).

![Mini 5 Pro wireframe](outputs/figures/wireframe_mini5pro.png)

*그림 1 — DJI Mini 5 Pro: 셰이딩(색=재질) / 와이어프레임 / 탑뷰. 지금 이 기체는 삼각형
29,824개 · 그룹 9개다(생성 시점 실측).
⚠ 그림 파일은 2026-08-16 13:17 판 렌더라 **프로펠러 평면형이 정본과 다르다** — 몸체(이 절의 주제)는
같고, 프롭 형상은 mesh05 를 볼 것. ← 그림 생성 report_mesh/src/viz_mesh_reports.py fig_wireframes().*

Mini 에서 볼 것 세 가지:

1. **전방 스윕 로터 배치** — `rotor_deg=(56.33, 132.47, 227.53, 303.67)`. 탑뷰에서 앞 모터가 45° 가 아니라
   56.3° 로 옆으로 벌어져 있다. 조사에서 확인한 로터 좌표 (±76, ±114) mm 그대로다(← src/drones.py).
2. **앞 모터가 낮다** — `rotor_z_mm=(-7.0, 7.0, 7.0, -7.0)`. 프롭 지름(152.4 mm)이
   앞뒤 모터 간격(152 mm)보다 커서 **프롭 디스크가 겹치는** 기체라, 실물은 앞 모터를 낮춰 충돌을
   피한다. 와이어프레임 패널에서 앞쪽 프롭 두 개가 뒤쪽보다 14 mm 낮게 앉은 게 보인다(← src/drones.py
   주석 — "실물은 앞 모터가 더 낮다(간섭 회피). 조사 확인").
3. **착륙장치가 전용 스키드가 아니라 모터 포드 밑 다리다** — `gear='motor_legs'`,
   다리 길이 31 mm. 앞쪽 두 모터 포드 아래에만 두 갈래 다리가 달리고 뒤쪽 포드에는 없다
   (← src/drones.py mini5pro note — 제품사진 계측, 밴드 ±15 %). 이 다리가 기체의 최저점이다.

색 규칙(모든 기종 공통): 플라스틱=밝은 회색(프로펠러 포함), 모터·배터리=파랑(금속), 카본(암)=검정, 카메라=주황,
PCB=초록 — **색이 곧 재질**이라 그림만 봐도 전파 물성이 읽힌다(← src/drones.py
MATERIAL_COLOR·drone_colors docstring).

![Phantom 4 wireframe](outputs/figures/wireframe_phantom4.png)

*그림 2 — DJI Phantom 4: 같은 3분할. 지금 이 기체는 삼각형 29,782개 ·
그룹 8개다(생성 시점 실측). 그림 파일은 그림 1 과 같은 2026-08-16 13:17 판 렌더다.
← 출처: viz_mesh_reports.py fig_wireframes().*

Phantom 에서 볼 것 세 가지:

1. **고정암 X자** — `fixed_arm=True`, `rotor_deg=(45, 135, 225, 315)` 의 대칭 X자.
   접이식과 달리 암이 굵고(0.085·diag) 덜 휜다(bend 0.02 vs 0.06). 암 그룹도 카본이 아니라
   동체와 같은 흰 셸(plastic)이다 — `arm_style='body'` 면 암을 body 그룹에 넣는다
   (← `src/drone_cad.py` 의 암 굵기·휨 분기).
2. **일체형 착륙다리** — `gear='legs'` → `_gear_arch`(뒤집힌 U 자 아치 다리 2개 + 바닥 레일).
   다리가 아래로 길게 뻗어, 접이식 소형기엔 없는 수직 구조물이 생긴다. 이게 높이 196 mm(공표 대각 350 mm 의 0.56 배 — 소비자 쿼드 중 가장 높은 비율)의 이유다.
3. **함몰 짐벌** — `gimbal_style='recessed'`. 같은 `_gimbal_hanging` 을 쓰되 동체에 바짝
   붙인다(cx 를 0.62 배로). 기수 아래 작은 비전센서 2개도 붙는다   (← `src/drone_cad.py` phantom4 짐벌 분기).

**두 그림을 나란히 두면** — 동체 단면 지수(3.2 vs 3.4), 코 처짐(0.18 vs 0.09), 암 굵기, 착륙장치
유무, 짐벌 위치가 전부 스펙 필드 몇 개에서 갈라져 나왔음을 볼 수 있다. 이것이 파라메트릭 CAD 의
요점이다: **개성은 데이터(스펙)에, 솜씨는 코드(조립 함수)에** 나눠 담긴다.

### §4.1 두 기종 숫자로 대비 (치수·면수는 생성 시점 실측 · 나머지는 DRONES)

| 항목 | Mini 5 Pro | Phantom 4 | 출처 |
|---|---|---|---|
| 대각(공식→실측) | 275.0 → 249.0 (-9.46%) | 350.0 → 350.0 (+0.00%) | 정본 판 실측(§3.1) |
| 높이(공식→실측) | 91.0 → 91.0 (-0.00%) | 196.0 → 196.0 (+0.00%) | 〃 |
| 프롭 지름(공식→실측) | 152.4 → 152.4 (-0.00%) | 240.0 → 240.0 (-0.00%) | 〃 |
| 삼각형 수 | 29,824 | 29,782 | 〃 |
| 그중 프로펠러 | 14,000 (46.9 %) | 13,920 (46.7 %) | 〃 |
| 그룹 수 | 9 | 8 | 〃 |
| 이륙중량 | 249.9 g | 1380 g | src/drones.py DRONES·docs/SPECS.md |
| 암/착륙장치 | 접이식 · 모터 포드 아래 다리(31 mm) | 고정암 · 스키드 다리 | src/drones.py,170 |
| 로터 z 오프셋 | (-7.0, 7.0, 7.0, -7.0) mm | 없음 | src/drones.py |

그룹 수가 9 대 8 로 갈리는 것은 **`accent`(전방 식별용 소형 파트 — 재질=플라스틱이라 셸과 같은 회색) 한 그룹 때문**이다 —
Mini 에는 있고 Phantom 에는 없다. `gear`(착륙장치)는 둘 다 있다(위 코드 셀 출력 참조).
참고로 Mavic 4 Pro 도 accent 가 없어 8개다.

⭐ 함대 전체로는 삼각형 **301,506개 · 꼭짓점 151,327개**다(정본 판, 생성 시점 실측).

## §5. 좌우대칭 — 설계 이유와 실측 검증

**왜 좌우대칭인가.** 멀티로터는 무게중심이 로터 배치의 중심에 있어야 호버링이 안정된다.
그래서 조립 규약이 코드에 명시돼 있다(← 출처: src/drones.py `motor_angles` docstring):

> "rotor_deg 는 좌우대칭이고 마주보는 쌍이 180° → 대각거리 스펙 보존 + 무게중심 중앙(비행안정)"

Mini 의 전방 스윕 배치 (56.33, 132.47, 227.53, 303.67) 는 그 규약의 **앞쪽 절반만** 지킨다.
좌우대칭은 성립한다 — 56.33°/303.67° 와 132.47°/227.53° 가 각각 xz 평면 거울상이다.
그러나 마주보는 쌍은 정확히 180° 가 **아니다**: 56.33°↔227.53° 는 171.2° 차이다.
로터가 정사각형이 아니라 **사다리꼴**로 놓여 있기 때문이고(앞 트랙이 뒤 트랙보다 넓다),
그래서 이 기체만 «대각거리 스펙 보존» 이 성립하지 않는다 — 마주보는 로터 거리는
248.26 mm 로 공표 대각 275 mm 보다 -9.7 % 작다.
⭐ 이것은 결함이 아니라 **선언된 선택**이다(스펙 `note` 가 그렇게 적는다). 다만 §3.1 표의 대각 오차가
그 선택의 결과라는 것, 그리고 **로터 간격을 인용할 때는 공표 대각이 아니라 축간거리를 써야 한다는 것**을
함께 읽어야 한다 ← 출처: `outputs/mesh_inspect_body_arms_0816.json` `findings` B12.
나머지 9기종은 마주보는 쌍이 정확히 180° 라 규약 그대로다.

**실측 검증(B_symmetry).** 완성 메쉬 표면에서 점을 뽑아 y→−y 로 뒤집은 점 구름과의 챔퍼 거리를
잰다 — 완벽 대칭이면 0 이 된다(← 출처: `report_mesh/outputs/mesh_verify_canon_0817.json` §B_symmetry, 측정 코드는
`report_mesh/src/verify_mesh_suite.py` 의 `sec_B_symmetry` 그대로). 단위는 mm:

| 기종 | 표본점 수(frame) | frame p50 | frame p95 | frame max | (참고) full p95 | 판정 p95≤2mm |
|---|---|---|---|---|---|---|
| DJI Mini 5 Pro | 57,646 | 0.11 | 1.37 | 1.99 | 14.3 | PASS |
| DJI Mavic 4 Pro | 117,420 | 0.25 | 1.68 | 2.02 | 27.4 | PASS |
| DJI Matrice 4E | 116,747 | 0.12 | 1.59 | 5.80 | 26.7 | PASS |
| DJI S1000+ | 1,301,187 | 0.09 | 1.31 | 2.10 | 24.1 | PASS |
| DJI Phantom 4 | 188,525 | 0.25 | 1.45 | 2.02 | 12.8 | PASS |
| Yuneec Typhoon H (H480) | 375,783 | 0.04 | 1.17 | 1.99 | 20.0 | PASS |
| Holybro X500 V2 | 509,895 | 0.01 | 0.94 | 2.00 | 4.1 | PASS |
| DJI Phantom 3 Professional | 135,261 | 0.17 | 1.53 | 2.00 | 20.8 | PASS |
| DJI Matrice 350 RTK | 671,902 | 0.22 | 1.61 | 2.06 | 41.8 | PASS |
| DJI Mini 2 | 43,673 | 0.15 | 1.24 | 2.00 | 11.2 | PASS |

프레임(비회전부)은 전 기종 **p95 ≤ 2 mm** — 최악이 DJI Mavic 4 Pro 의
1.68 mm 다. 남는 잔차는 대칭이 아닌 부품(레이저 측거처럼 한쪽에만 있는
센서, 짐벌 요크 디테일)과 표본 추출 노이즈다.

**full(프로펠러 포함) p95 는 4~42 mm 로 크다
— 버그가 아니다.** 프로펠러는 장착 위상(+12° 오프셋 ← src/drones.py)과 교대 회전 방향으로
앉아 있어 어느 순간의 스냅샷도 좌우 거울상이 아니다. 실물도 마찬가지다. 그래서 대칭 검증은
**frame_only(프레임만)** 로 판정한다 — full 수치는 '프롭이 대칭을 깨는 정도'의 참고값으로만 싣는다.

In [ ]:
# 좌우대칭 실측 — 정본 판 원장 §B_symmetry 를 그대로 표로. 판정: frame p95 ≤ 2 mm
print(f"{'key':10s} {'frame p50':>10s} {'frame p95':>10s} {'frame max':>10s} {'full p95':>9s}")
for k in V['_meta']['drones']:
    fo = V['B_symmetry'][k]['frame_only']['chamfer_mm']
    fu = V['B_symmetry'][k]['full']['chamfer_mm']
    print(f"{k:10s} {fo['p50']:10.2f} {fo['p95']:10.2f} {fo['max']:10.2f} {fu['p95']:9.1f}")

assert all(V['B_symmetry'][k]['frame_only']['chamfer_mm']['p95'] <= 2.0
           for k in V['_meta']['drones'])
print()
print('PASS — 전 기종 프레임 좌우대칭 p95 ≤ 2 mm (full 은 프로펠러 위상 때문에 원래 크다)')

## 정리

1. **스펙과 스타일을 분리했다** — 공식 숫자(출처·신뢰도 부착)는 물리로, 눈대중 파라미터는
   실루엣으로만 들어간다. 추정값(Mini 대각, Mavic 대각 모순)은 note 에 그대로 남겼다(§1).
2. **조립은 4가지 기법의 반복이다** — 초타원 로프트(동체·캐노피), 베지에 스윕(암·다리),
   회전체(모터 벨·RTK 돔), 불리언 합집합(내부 면 제거 — 그룹 **사이**와 그룹 **안** 둘 다).
   짐벌 3종·착륙장치 3종·내부 battery/pcb 가 기종 개성과 레이더 물리를 담당한다(§2).
3. **공식 외형이 최종 기준이다 — 다만 그 기준이 닿는 데까지다.** envelope fit 후 치수 최악 오차
   9.46% (DJI Mini 5 Pro). 공식값을 강제한 축은 오차 0 인데, 그 0 은 «검증» 이 아니라
   **«구성상 보장»** 이다 — 어느 축을 강제하는지는 기체마다 다르다(§3.1·§3.2, 자세히는 mesh08 §1).
   공표 외형이 말하지 않는 부품 치수는 **바깥 참값**이 따로 보고, 지금 그중 24행이
   어긋나 있다(§3.1b).
4. **좌우대칭은 규약 + 실측으로 보증한다** — 프레임 p95 ≤ 2 mm 전 기종 통과(§5).

**현재 남은 한계** — 있는 그대로 적는다:

| 무엇 | 기체 | 지금 이만큼 | 어디에 실리나 |
|---|---|---|---|
| 공표 높이를 **형상이 아니라 세로 배율**로 맞춘다 | mini5pro · mavic4pro | 세로 배율 1.2985 / 1.5977 — 형상표의 셸 높이 45.05 / 62.10 mm 가 메쉬에서 60.00 / 103.61 mm 로 나온다 | 전 부품이 같은 배율로 늘어난다(§3.2). σ 상한은 배율이 바뀌었으므로 재측정 대상 |
| 뜬 파트(기체에 안 닿는 부품) | phantom4 · phantom3 · m350rtk · x500v2 | 착륙아치 8.3~8.5 / 13.7~13.8 mm · 프롭 허브 6.0 mm · 레일 4.0 mm | 간극 0.05~0.16 λ @3.5 GHz — 면적은 그대로고 가림·다중반사·위상이 바뀐다 |
| 로터면이 공식 CAD 보다 위 | matrice4e | 18.5 mm = 0.216 λ @3.5 GHz. 명세가 F19~F21 로 «엔진 변경 필요» 라 미뤄 둔 자리 | 프롭 장착 높이가 함께 움직인다 ⏳ |
| 짐벌이 세 조각으로 떨어져 있다 | phantom3 | 방진판·요 샤프트·카메라 블록이 서로 2.99~8.18 mm 씩 벌어져 있고, 기체 표면과도 2.44~37.67 mm 떨어져 있다 — 잇는 구조가 없다 | 면적 360 cm² 가 공중에 뜬다. 가림·다중반사가 달라지고, PO 와 SBR 이 같은 메쉬를 다르게 읽는다 |
| 짐벌 헬퍼의 «선언 치수» ↔ 실제 크기 | mini5pro·matrice4e·phantom4·s1000plus 등 | 인자로 넣은 상자 위에 요크·렌즈·마운트가 더 붙어 최대 1.57 배로 지어진다 | 인자를 실물 치수로 인용하면 틀린다. mini2 처럼 **역산**해야 실물과 맞는다 |
| 카본 판이 `plastic` 그룹에 있다 | s1000plus | 판 2장만 세도 body 합집합 전 면적의 69.3 % (스탠드오프 기둥까지 넣은 스택은 74.5 %) | 면 반사율 +10.14 dB (carbon 0.90 ↔ plastic 0.28) |

← 출처: `outputs/mesh_inspect_body_arms_0816.json` `findings` · `outputs/mesh_inspect_gimbal_sensors_0816.json` `_summary` · `outputs/mesh_inspect_materials_check_0816.json` `findings` · 세로 배율은 `report_mesh/outputs/mesh_verify_canon_0817.json`, 셸 높이는 생성 시점 실측. 위 표의 dB 는 대부분 **평판극한 상한**이지 커널이 계산한 σ 가 아니다.

거기에 더해: 내부 battery/pcb 치수는 기종별 실물이 비공개라 동체 비율 추정이고,
비등방 배율 때문에 모터 원통이 약간 타원이다.

### ⭐ 이 편이 «장담하지 못하는» 것 — 인증서가 선언한 경계

위 표는 «알고 있는 결함» 이다. 그것과 별개로, **검사가 원리적으로 못 보는 자리**가 있다.
메쉬 인증서가 그것을 선언해 두었고, 이 편의 주제(부품을 조립해 몸체를 만든다)에 바로 걸리는
항목은 셋이다 ← 출처: `docs/MESH_CERTIFICATE.md` §3.1~§3.3:

| 무엇을 못 보나 | 크기 | 뜻 |
|---|---|---|
| **부품이 통째로 빠져도 검사가 조용하다** | mini5pro 에서 `canopy`(표면적 8.4 %)·`gear`(2.1 %)를 지워도 검사 10계열과 바깥 참값 7행이 전부 통과 | «있어야 할 부품 명세» 라는 것이 없다. 조립이 한 부위를 빠뜨려도 자동으로는 안 잡힌다 |
| **출하된 파일 자체는 검사된 적이 없다** | 검사는 메모리 배열에서 돌고 파일은 그 뒤에 쓰인다 — 1 µm 반올림 때문에 되읽으면 2/10 기체가 자기 검사에 실패한다 | 물리 영향은 무시할 만하지만 «잣대 0» 은 **메모리 안의 메쉬**에 대한 말이다 |
| **실물 충실도** | 바깥 참값 77행 중 24행 어긋남(§3.1b) · 독립 참값이 0행인 기체 2종 | 이 편이 보증하는 것은 «스펙대로 지었다» 이지 «실물과 같다» 가 아니다 |

### 지금 «모른다» 고 선언한 것

- mavic4pro 의 세로 예산 35.2 mm 가 **어디에** 있어야 하는지 못 정했다. 공표 언폴드 높이 135.2 mm 는 공식이지만 그 135.2 를 다리·셸·짐벌·모터에 어떻게 나누는지는 사진 한 장으로 안 풀린다 — matrice4e 처럼 공식 CAD 가 필요하고 DJI 는 Mavic 4 Pro CAD 를 공개하지 않는다.
- mini5pro 셸 높이의 1차 출처가 없다. `fh = 0.495` 는 공표 높이(91, **프롭 포함**)에 대한 비율이고, 그 91 자체가 프롭을 포함하므로 셸 높이를 직접 구속하지 않는다. 폴디드 68 mm 로 교차검산하려면 짐벌 매달림 길이를 따로 재야 하는데 그 값도 실측이 없다.
- B3 의 «기수 정면 정반사 10 dB» 는 평판극한 상한일 뿐 커널 결과가 아니다. 진짜 값을 알려면 스무딩 0/4 두 메쉬로 PO 를 돌려야 하는데 이 라운드는 σ 파일을 열지 않았다.
- phantom3·phantom4 착륙아치가 «어디에» 붙어야 하는지 — 매뉴얼 정면도가 붙는 곳 좌우 스팬은 주지만 앞뒤 부착점은 셸 곡면과의 교선이라 표에서 못 읽는다.
- x500v2 배터리 트레이 2.65 mm 는 2026-08-04 원장이 «면-대-면 접촉의 거짓양성» 이라 적었는데 양방향 잣대로도 남는다. 어느 쪽이 맞는지 판정하지 않았다.
- **프로펠러의 두께** — 시위·팁·기체별 평면형은 정해졌지만(mesh05), 두께는 10기종 중 **2기종만** 근거가 있다(mini2 절대 mm · typhoonh480 t/c). 나머지는 빈칸이고, 이유는 «다음 라운드» 가 아니라 **사진으로는 원리적으로 못 잰다** 는 것이다 — 겉보기 높이에 시위가 두께의 몇 배로 섞여 들어온다.
- **배터리 재질** — ⚠ **미해결로 선언한다.** 지금 고치지 않는 이유: 셀 스택의 실제 치수가 1차 출처 0 이고, 추정으로 줄이면 «측정 아닌 값» 을 또 하나 심는다. 대신 **모든 절대 σ 인용에 «배터리는 팩 외피 전체를 금속으로 본 값(상한 쪽 1~3 dB)» 단서를 붙일 것.**
- **카메라 재질 0.85 의 출처** — docs/MATERIAL_SOURCES.md §6-4 가 이미 «출처 없음 · 총 σ 를 최대 1.81 dB 움직임» 으로 적어 뒀다 (그 1.81 은 mavic4pro·1.843 GHz 한 조건에서 잰 값이다).

⭐ **빈칸이 가짜 값보다 낫다.** 위 항목들은 값을 채워 넣는 대신 비워 두었다.

## 재현 명령

```bash
cd /workspace/sionna/report_mesh
# 1) 정본 판 원장 재생성 — 스위치를 안 주면 정본이다(약 3~5분, CPU)
/workspace/.venvs/py312/bin/python src/verify_mesh_canon_0817.py   # A/B/C/D/F/G
/workspace/.venvs/py312/bin/python src/mesh_canon_0817.py          # 부품·예산
# 2) 그림(outputs/figures/*.png) 재생성
/workspace/.venvs/py312/bin/python src/viz_mesh_reports.py
# 3) 이 노트북 재생성 (부품 높이는 이때 정본 판으로 직접 잰다)
/workspace/.venvs/py312/bin/python src/make_mesh04.py

# ⛔ 옛 판(정본 전환 전)을 비트동일하게 되살리려면:
MESH_FIX=none BLADE_LAW=legacy /workspace/.venvs/py312/bin/python src/verify_mesh_suite.py
```

⭐ 이 편의 수는 **정본 판**(`MESH_FIX=battery,i5` · `BLADE_LAW=per_airframe`)의 것이다.
계산 산출물은 파일 이름에 꼬리표 `_mfixbatteryi5_blperairframe` 가 붙어 두 판이 섞이지 않는다.

**다음 리포트** — mesh05: 프로펠러 편(진짜 익형 블레이드 — NACA 단면·모델별 기하 피치·시미터
스윕, 그리고 지름을 «맞춰 놓는» 정규화가 무엇을 보증하고 무엇을 안 보증하는지).
같은 폴더의 `mesh05_*.ipynb`.